# Number Theory and Cryptography Workshop

## Md.Samiul Haque Ruf

## ID: IT-23011

Topics: Divisibility and Modular Arithmetic, GCD and Extended Euclid, LCM, Diophantine Equations, Modular Inverse, Fast Modular Exponentiation, CRT, Fermat's Theorem, Euler's Totient, Primes and Primality Tests.

In [ ]:
import math, random

## 1. Divisibility and Modular Arithmetic
- $b \mid a$ means $a = bk$ for some integer $k$.
- Division with remainder: $a = qb + r,\ 0 \le r < b$.
- $a \equiv b \pmod m \iff m \mid (a-b)$. Addition and multiplication keep congruence.
- Digit-sum rule: $10 \equiv 1 \pmod 3$, so a number is congruent to its digit sum mod 3.

In [ ]:
print(divmod(15, 4), divmod(-13, 3))              # (quotient, remainder)
print((17*(12*19+5) - 23) % 3)                    # 2
print(pow(99, 99, 100))                           # last two digits of 99^99 -> 99
num = 3475
print(num % 3, sum(map(int, str(num))) % 3)       # digit-sum rule

## 2. GCD and Euclid's Algorithm
$\gcd(a,b) = \gcd(b,\ a \bmod b)$. Each step shrinks a number by at least half, so it is very fast.

In [ ]:
def gcd(x, y):
    assert x >= y >= 0 and x + y > 0
    return gcd(y, x % y) if y > 0 else x

print(gcd(24, 16), gcd(17, 9), gcd(239, 0))
print(gcd(1849639579327, 790933790547))

## 3. Extended Euclid
Finds $d = \gcd(a,b)$ and integers $x, y$ with $d = ax + by$.
If $d = bp + (a \bmod b)q$ then $x = q,\ y = p - \lfloor a/b \rfloor q$.

In [ ]:
def extended_gcd(m, n):
    assert m >= n >= 0 and m + n > 0
    if n == 0:
        g, u, v = m, 1, 0
    else:
        g, s, t = extended_gcd(n, m % n)
        u, v = t, s - (m // n) * t
    assert m % g == 0 and n % g == 0 and g == m * u + n * v
    return g, u, v

print(extended_gcd(8, 4))
print(extended_gcd(27, 15))
print(extended_gcd(391, 299))      # 23 = 391*(-3) + 299*4

## 4. LCM
$$\operatorname{lcm}(a,b) = \frac{ab}{\gcd(a,b)}$$

In [ ]:
def lcm(x, y):
    return x // gcd(max(x, y), min(x, y)) * y

assert lcm(9, 3) == 9 and lcm(17, 37) == 629
print(lcm(24, 16), lcm(9, 17))

## 5. Diophantine Equations
$ax + by = c$ has integer solutions $\iff \gcd(a,b) \mid c$.
Get $(x_0, y_0)$ by scaling the Extended Euclid result by $c/d$.
All solutions: $x = x_0 + tq,\ y = y_0 - tp$ where $a = dp,\ b = dq$.

In [ ]:
def diophantine(p, q, r):
    if p >= q:
        g, u0, v0 = extended_gcd(p, q)
    else:
        g, v0, u0 = extended_gcd(q, p)
    assert r % g == 0, "no integer solution"
    k = r // g
    u, v = u0 * k, v0 * k
    assert p * u + q * v == r
    return u, v

print(diophantine(34, 51, 170))
print(diophantine(91, 117, 65))
print(diophantine(187, 55, 121))
# diophantine(187, 55, 45) -> AssertionError, because gcd = 11 does not divide 45

## 6. Modular Inverse and Division
$a^{-1} \bmod n$ exists $\iff \gcd(a,n)=1$. From $nx + ay = 1$ we get $a^{-1} \equiv y \pmod n$.
Then $b/a \equiv b \cdot a^{-1} \pmod n$.

In [ ]:
def modInverse(val, mod):
    val %= mod
    g, _, inv = extended_gcd(mod, val)
    assert g == 1, "inverse does not exist"
    return inv % mod

def divide(num, den, mod):
    assert mod > 1 and den > 0
    return (modInverse(den, mod) * num) % mod

print(divide(9, 5, 3), divide(223, 67, 54))
print(divide(7, 2, 9))                                   # 8
print([k for k in range(1, 6) if math.gcd(k, 6) == 1])   # only 1 and 5 are invertible mod 6

## 7. Fast Modular Exponentiation
Write $e$ in binary and use repeated squaring:
$b^e = \prod_{e_i=1} b^{2^i} \pmod m$. It needs at most $2\log_2 e$ multiplications.

In [ ]:
def FastModularExponentiation(base, exp, mod):
    acc, cur = 1 % mod, base % mod
    while exp > 0:
        if exp & 1:
            acc = (acc * cur) % mod
        cur = (cur * cur) % mod
        exp >>= 1
    return acc

print(FastModularExponentiation(4, 5, 3))
print(FastModularExponentiation(7, 13, 11))              # 2
tests = [(random.randint(1, 10**6), random.randint(0, 10**6), random.randint(2, 10**6)) for _ in range(500)]
assert all(FastModularExponentiation(bs, ex, mo) == pow(bs, ex, mo) for bs, ex, mo in tests)

## 8. Chinese Remainder Theorem
If $\gcd(a,b)=1$, then $n \equiv r_a \pmod a,\ n \equiv r_b \pmod b$ has a unique solution mod $ab$.
With $ax + by = 1$:
$$n = r_a \cdot by + r_b \cdot ax \pmod{ab}$$

In [ ]:
def ChineseRemainderTheorem(m1, r1, m2, r2):
    if m1 >= m2:
        g, s, t = extended_gcd(m1, m2)       # m1*s + m2*t = 1
    else:
        g, t, s = extended_gcd(m2, m1)
    assert g == 1
    return (r1 * m2 * t + r2 * m1 * s) % (m1 * m2)

print(ChineseRemainderTheorem(13, 7, 6, 5))      # 59
print(ChineseRemainderTheorem(28, 20, 19, 3))    # 440

## 9. Fermat's Little Theorem and Euler's Theorem
- Fermat: $p$ prime, $p \nmid a \Rightarrow a^{p-1} \equiv 1 \pmod p$. Also $a^n \equiv a^{n \bmod (p-1)} \pmod p$.
- Euler's totient $\phi(n)$ = count of numbers in $[0,n-1]$ coprime to $n$. $\phi(p)=p-1$, $\phi(pq)=(p-1)(q-1)$.
- Euler: $\gcd(a,n)=1 \Rightarrow a^{\phi(n)} \equiv 1 \pmod n$.

In [ ]:
def phi(n):
    return sum(1 for k in range(n) if math.gcd(k, n) == 1) if n > 1 else 1

print([phi(n) for n in range(1, 13)])
print(phi(55), 4 * 10)

for p in [5, 7, 11, 101]:
    assert all(pow(a, p - 1, p) == 1 for a in range(1, p))
for n in [10, 15, 55, 100]:
    assert all(pow(a, phi(n), n) == 1 for a in range(1, n) if math.gcd(a, n) == 1)
print("Fermat and Euler verified")

## 10. Primes
- Infinitely many primes (Euclid): $p_1 p_2 \cdots p_k + 1$ has a new prime factor.
- Prime Number Theorem: $\pi(x) \sim x / \ln x$.
- Sieve of Eratosthenes: strike out multiples of primes up to $\sqrt{n}$.

In [ ]:
def sieve(limit):
    flags = [True] * (limit + 1)
    flags[0] = flags[1] = False
    for i in range(2, int(limit**0.5) + 1):
        if flags[i]:
            for j in range(i*i, limit + 1, i):
                flags[j] = False
    return [i for i, ok in enumerate(flags) if ok]

print(sieve(60))
primes = sieve(100000)
print(len(primes), round(100000 / math.log(100000)))      # pi(x) vs x/ln x

## 11. Primality Tests
**Fermat test:** if $a^{n-1} \not\equiv 1 \pmod n$ then $n$ is composite. Composites that pass are *pseudoprimes*.
**Carmichael numbers** pass for every coprime base (smallest: 561). Korselt: $n$ is Carmichael iff it is composite,
squarefree and $(p-1) \mid (n-1)$ for every prime $p \mid n$.

In [ ]:
def fermat_test(n, base): return pow(base, n - 1, n) == 1

print(fermat_test(341, 2), fermat_test(341, 3))   # True False  (341 is a base-2 pseudoprime)
print(fermat_test(91, 2), fermat_test(91, 3))     # False True

def prime_factors(n):
    factors, d = [], 2
    while d * d <= n:
        while n % d == 0:
            factors.append(d)
            n //= d
        d += 1
    if n > 1:
        factors.append(n)
    return factors

def is_carmichael(n):
    fs = prime_factors(n)
    return len(fs) > 1 and len(set(fs)) == len(fs) and all((n - 1) % (p - 1) == 0 for p in fs)

print([n for n in range(2, 3000) if is_carmichael(n)])     # 561, 1105, 1729, 2465, 2821
print(all(fermat_test(561, a) for a in range(2, 561) if math.gcd(a, 561) == 1))

## 12. Mersenne Primes and Perfect Numbers
$M_p = 2^p - 1$. **Lucas-Lehmer:** $l_1 = 4,\ l_n = l_{n-1}^2 - 2$; for odd prime $p$, $M_p$ is prime $\iff M_p \mid l_{p-1}$.
If $2^n - 1$ is prime, then $2^{n-1}(2^n - 1)$ is a perfect number (equals the sum of its proper divisors).

In [ ]:
def lucas_lehmer(p):
    if p == 2: return True
    mersenne, term = 2**p - 1, 4
    for _ in range(p - 2):
        term = (term * term - 2) % mersenne
    return term == 0

candidates = sieve(130)
exps = [p for p in candidates if lucas_lehmer(p)]
print(exps)                                        # 2 3 5 7 13 17 19 31 61 89 107 127
print([2**(p-1) * (2**p - 1) for p in exps[:4]])   # 6 28 496 8128